# 01 · First look at the battle data

Run `python -m clashlab.collect` a few times, then `python -m clashlab.flatten`, before running this notebook.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from clashlab import config

df = pd.read_parquet(config.PROCESSED_DIR / 'battles.parquet')
print(f'{len(df):,} battles from {df.battle_time.min():%Y-%m-%d} to {df.battle_time.max():%Y-%m-%d}')
df['type'].value_counts()

30 battles from 2026-10-12 to 2026-10-13


type
pathOfLegend    30
Name: count, dtype: int64

## Keep ranked/ladder 1v1 only
Other modes (challenges, clan wars, friendlies) have different rules and card levels, so they'd muddy the win-rate analysis.

In [2]:
ranked = df[df['type'].isin(['pathOfLegend', 'PvP'])].dropna(subset=['team_won']).copy()
print(f'{len(ranked):,} ranked battles')
ranked[['team_starting_trophies', 'opp_starting_trophies']].describe()

30 ranked battles


,team_starting_trophies,opp_starting_trophies
count,0,0
unique,0,0
top,NaN,NaN
freq,NaN,NaN


## Card usage and raw win rate
Each battle has two decks, so stack both sides. Raw win rates are confounded by player skill and card levels; separating those out is the point of the modelling notebooks.

In [ ]:
team = ranked[['team_cards', 'team_won']].rename(columns={'team_cards': 'cards', 'team_won': 'won'})
opp = ranked[['opp_cards']].rename(columns={'opp_cards': 'cards'}).assign(won=1 - ranked['team_won'])
decks = pd.concat([team, opp], ignore_index=True).explode('cards')

card_stats = (decks.groupby('cards')['won']
                   .agg(games='size', win_rate='mean')
                   .assign(use_rate=lambda d: d.games / (2 * len(ranked)))
                   .sort_values('use_rate', ascending=False))
card_stats.head(20)

,games,win_rate,use_rate
cards,,,
Barbarian Barrel,30,0.666667,0.500000
Rascals,30,0.633333,0.500000
Ice Wizard,25,0.680000,0.416667
Fireball,24,0.708333,0.400000
Skeleton Barrel,24,0.666667,0.400000
Mortar,21,0.761905,0.350000
Minions,21,0.714286,0.350000
Cannon Cart,20,0.750000,0.333333
Skeletons,17,0.411765,0.283333


: 

In [ ]:
top = card_stats[card_stats.games >= 200]
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(top.use_rate, top.win_rate, alpha=0.6)
for name, r in top.head(15).iterrows():
    ax.annotate(name, (r.use_rate, r.win_rate), fontsize=8)
ax.axhline(0.5, color='grey', lw=0.8)
ax.set_xlabel('Use rate'); ax.set_ylabel('Raw win rate'); ax.set_title('Card meta: popularity vs raw win rate')
plt.show()

## Selection bias check
`team` is always the crawled player, so team win rate may not be 50%. Worth knowing before modelling.

In [ ]:
print(f"Crawled-player win rate: {ranked.team_won.mean():.3f}")